# IMPORTS

In [2]:
import os
import sys
import pandas as pd
import seaborn as sn
import numpy as np
import json
import matplotlib.pyplot as plt
from datetime import datetime

## Ativar o ambiente virtual

In [3]:
# Caminho para o ambiente virtual
venv_path = os.path.join('.', 'venv_health', 'Scripts', 'activate')  # Ajuste para Linux/Mac: 'bin/activate_this.py'

os.system(venv_path)

# Agora o script usará as dependências do ambiente virtual
print("Ambiente virtual ativado:", sys.prefix)

Ambiente virtual ativado: d:\python\data_health\venv_health


# Tratamento dos dados

## Buscando os arquivos já tratados

Os dados são obtidos de um arquivo csv já tratado, caso não tenha o arquivo ele será criado automaticamente pelo script que executará ambas etapas:

- Converter o arquivo .dbc em .csv, gerando o arquivo `HANSENIASE_TOTAL_`

- Pré-processamento dos dados do Wsus¹ gerando o csv já tratado `HANSENIASE_PROCESS_`

In [4]:
# Path dos arquivos a serem analisados
path_csv = os.path.join('.', 'data', 'HANS')
arquivos = []
for f in os.listdir(path_csv):
    if f.startswith("HANSENIASE_TOTAL_") and f.endswith(".csv"):
        try:
            # Extrair data do nome do arquivo
            data_str = f.split('_')[-3:]  # Pega os últimos 3 elementos (dd, mm, yyyy)
            data = datetime.strptime('_'.join(data_str).replace('.csv', ''), '%d_%m_%Y')
            arquivos.append((data, f))
        except Exception as e:
            print(f"Arquivo com formato inválido: {f} - {e}")

# Executar script externo se nenhum arquivo for encontrado
if not arquivos:
    print("Nenhum arquivo encontrado. Executando script para gerar arquivo...")
    path_convert_dbc = os.path.join(".\\","convert_dbc.py")
    resultado = os.system('python ' + path_convert_dbc)
    
    if resultado == 0:
        print("Script executado com sucesso. Verificando novo arquivo...")
        # Recarregar lista de arquivos após execução do script
        for f in os.listdir(path_csv):
            if f.startswith('HANSENIASE_TOTAL_') and f.endswith('.csv'):
                try:
                    data_str = f.split('_')[-3:]
                    data = datetime.strptime('_'.join(data_str).replace('.csv', ''), '%d_%m_%Y')
                    arquivos.append((data, f))
                except Exception as e:
                    print(f"Erro ao processar novo arquivo: {f} - {e}")
    else:
        print("Erro na execução do script. Arquivo não gerado.")

if arquivos:
    # Ordenar arquivos pela data mais recente
    arquivos.sort(reverse=True, key=lambda x: x[0])
    
    # Pegar arquivo mais recente
    ultimo_arquivo = os.path.join(path_csv, arquivos[0][1])
    
    print(f"Carregando arquivo mais recente: {ultimo_arquivo}")
    
    # Carregar dados
    df_dados = pd.read_csv(ultimo_arquivo, encoding='utf-8', low_memory=False)
    
    print("\nPrimeiras linhas do DataFrame:")
    print(f"\nTotal de registros: {len(df_dados):,}")
    print(f"Data de referência: {arquivos[0][0].strftime('%d/%m/%Y')}")
else:
    print("Nenhum arquivo válido encontrado na pasta")
   
df_dados.head()  

Carregando arquivo mais recente: .\data\HANS\HANSENIASE_TOTAL_06_03_2025.csv

Primeiras linhas do DataFrame:

Total de registros: 984,168
Data de referência: 06/03/2025


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_DIAG,SEM_DIAG,...,AVAL_ATU_N,ESQ_ATU_N,DOSE_RECEB,EPIS_RACIO,DTMUDESQ,CONTEXAM,DTALTA_N,TPALTA_N,IN_VINCULA,NU_LOTE_IA
0,2,A309,2001-01-10,2001,41.0,410304,1359.0,2741474.0,2000-12-10,NaN,...,0,3,NaN,NaN,NaN,0.0,2002-10-02,1,0.0,0000000
1,2,A309,2001-01-17,2001,41.0,410940,1359.0,2741369.0,2000-12-23,NaN,...,0,3,NaN,NaN,NaN,2.0,2001-12-19,1,0.0,0000000
2,2,A309,2001-01-16,2001,41.0,410940,1359.0,2741369.0,2001-01-16,NaN,...,0,1,NaN,NaN,NaN,1.0,2001-06-27,1,0.0,0000000
3,2,A309,2001-01-08,2001,41.0,411780,1359.0,2743116.0,2001-01-08,NaN,...,0,3,NaN,NaN,NaN,3.0,2002-01-28,1,0.0,0000000
4,2,A309,2001-01-02,2001,41.0,411780,1359.0,2743116.0,2000-01-11,NaN,...,2,3,NaN,NaN,NaN,0.0,2002-03-14,1,0.0,0000000


Verificando os dados do Ano de notificação

In [5]:
df_dados['NU_ANO']
df_dados['NU_ANO'].head()

0    2001
1    2001
2    2001
3    2001
4    2001
Name: NU_ANO, dtype: int64

### Retirar a linha de Total

In [6]:
# Remover as linhas onde TP_NOT é 'Total'
df_dados = df_dados[df_dados['TP_NOT'] != 'Total']
print(f"DataFrame atualizado com {df_dados.shape[0]} linhas.")

DataFrame atualizado com 984168 linhas.


### Retirar duplicadas pela váriavel NDUPLIC_N

In [7]:
# Filtrar o DataFrame excluindo as linhas com NDUPLIC_N == 2
df_dados = df_dados[df_dados["NDUPLIC_N"] != 2]

print("Linhas com NDUPLIC_N == 2 foram removidas.")
print(f"DataFrame atualizado com {df_dados.shape[0]} linhas.")

Linhas com NDUPLIC_N == 2 foram removidas.
DataFrame atualizado com 984035 linhas.


### Colunas com valor constante

In [8]:
# Verificar colunas com valor constante e seus valores
colunas_constantes = {col: df_dados[col].iloc[0] for col in df_dados.columns if df_dados[col].nunique() == 1}

if colunas_constantes:
    print("Colunas com valor constante e seus respectivos valores:")
    for coluna, valor in colunas_constantes.items():
        print(f"{coluna}: {valor}")
else:
    print("Não há colunas com valores constantes no DataFrame.")


Colunas com valor constante e seus respectivos valores:
TP_NOT: 2
ID_AGRAVO: A309
CS_FLXRET: nan
FLXRECEBI: nan


### Retirar as colunas com valor constante

In [9]:
# Dropar as colunas com valor constante
df_dados = df_dados.drop(columns=colunas_constantes)

print(f"Colunas removidas: {colunas_constantes}")
print(f"DataFrame atualizado com {df_dados.shape[1]} colunas.")

Colunas removidas: {'TP_NOT': 2, 'ID_AGRAVO': 'A309', 'CS_FLXRET': nan, 'FLXRECEBI': nan}
DataFrame atualizado com 59 colunas.


### Obter os valores únicos

In [10]:
### Investigação de variaveis
# Obter os valores únicos
valores_unicos = df_dados["NU_LOTE_IA"].unique()
valores_unicos[5]

'2007003'

### Tratando a varivel CS_SEXO

In [27]:
df_dados['CS_SEXO_CAT'] = df_dados['CS_SEXO'].replace({np.nan: 9, 'M': 1, 'F': 2, 'I': 9}).astype(int)
df_dados['CS_SEXO_CAT'].value_counts()

CS_SEXO_CAT
1    550808
2    433041
9       186
Name: count, dtype: int64

### Tratamento a variavel `AVAL_ATU_N`

        0. grau zero

        1. grau I

        2. grau II

        3. Não avaliado

        9- Ignorado


In [12]:
# Converter a coluna 'CLASSATUAL' para string primeiro
df_dados["AVAL_ATU_N_CAT"] = df_dados['AVAL_ATU_N'].astype(str)

# Criar a nova coluna categorizada
df_dados["AVAL_ATU_N_CAT"] = df_dados['AVAL_ATU_N_CAT'].replace({'nan': '9', 'N': '9'}).astype(int)
df_dados["AVAL_ATU_N_CAT"].value_counts()

AVAL_ATU_N_CAT
0    406518
9    277218
3    168595
1     96641
2     35063
Name: count, dtype: int64

### Tratando a variavel CLASSATUAL 

Categorizar os valores 'N' e '9' como 0 na coluna 'CLASSATUAL' do DataFrame df_dados.

Na documentação somente possui os valores de 1 e 2:

        1. PB (Paucibacilar) 

        2. MB (Multibacilar) 

In [13]:
# Converter a coluna 'CLASSATUAL' para string primeiro
df_dados['CLASSATUAL'] = df_dados['CLASSATUAL'].astype(str)

# Criar a nova coluna categorizada
df_dados['CLASSATUAL_CAT'] = df_dados['CLASSATUAL'].replace({'nan': '0', 'N': '0', '9': '0'})
df_dados['CLASSATUAL_CAT'] = df_dados['CLASSATUAL_CAT'].map({'1': 1, '2': 2, '0': 0}).fillna(0).astype(int)
df_dados['CLASSATUAL_CAT'].value_counts()

CLASSATUAL_CAT
2    650016
1    331375
0      2644
Name: count, dtype: int64

### Tratamento da variavel idade

com Código (4 - Ano e 0 para completar) A composição da variável obedece o seguinte critério: 1° dígito: Ex: 3009 – nove meses, 4018 – dezoito anos

In [ ]:
def extrair_idade(valor):
    valor_str = str(valor).zfill(4)
    unidade = int(valor_str[0])
    quantidade = int(valor_str[1:])

    if unidade == 4:  # Anos
        return quantidade, 0
    elif unidade == 3:  # Meses
        if quantidade > 12:
            anos = quantidade // 12
            meses = quantidade % 12
            if meses == 0:  # Se não houver meses restantes, meses = 0
                return anos, 0
            else:
                return anos, meses
        else:
            return 0, quantidade
    elif unidade == 2:  # Dias
        return 0, 1  # Considerando que dias são menores que meses
    elif unidade == 1:  # Horas
        return 0, 1  # Considerando que horas são menores que dias
    else:
        return 0, 0

# Aplicar a função à coluna NU_IDADE_N
df_dados[['IDADE_ANOS', 'IDADE_MESES']] = df_dados['NU_IDADE_N'].apply(
    lambda x: pd.Series(extrair_idade(x))
)

### Tratamento variavel CS_GESTANT

In [ ]:
df_dados['CS_GESTANT_CAT'] = df_dados['CS_GESTANT'].replace({np.nan: 9}).astype(int)

df_dados['CS_GESTANT_CAT'].value_counts()

CS_GESTANT_CAT
6    608826
5    204663
9    166181
2      1382
0      1184
1      1006
4       793
Name: count, dtype: int64

### Tratamento variavel CS_RACA

In [26]:
df_dados['CS_RACA_CAT'] = df_dados['CS_RACA'].replace({np.nan: 9}).astype(int)

df_dados['CS_RACA_CAT'].value_counts()

CS_RACA_CAT
4    485235
1    256308
2    118802
9    108217
3     11750
5      3723
Name: count, dtype: int64

### Tratamento variavel NU_LESOES

In [ ]:
df_dados['NU_LESOES_CAT'] = df_dados['NU_LESOES'].replace(np.nan, 0).astype(int)

df_dados['NU_LESOES_CAT'].value_counts()

### Tratamento variavel FORMACLINI

In [ ]:
df_dados['FORMACLINI_CAT'] = df_dados['FORMACLINI'].replace({np.nan: 5, 0: 5}).astype(int)

df_dados['FORMACLINI_CAT'].value_counts()

### Tratamento variavel AVALIA_N

In [ ]:
df_dados['AVALIA_N_CAT'] = df_dados['AVALIA_N'].replace({np.nan: 3}).astype(int)

df_dados['AVALIA_N_CAT'].value_counts()

### Tratamento variavel CLASSOPERA

In [ ]:
df_dados['CLASSOPERA_CAT'] = df_dados['CLASSOPERA'].replace({np.nan: 3, 9: 3}).astype(int)

df_dados['CLASSOPERA_CAT'].value_counts()

### Tratamento variavel BACILOSCOP

In [ ]:
df_dados['BACILOSCOP_CAT'] = df_dados['BACILOSCOP'].replace({np.nan: 9}).astype(int)

df_dados['BACILOSCOP_CAT'].value_counts()

### Tratamento variavel ESQ_INI_N

In [ ]:
df_dados['ESQ_INI_N_CAT'] = df_dados['ESQ_INI_N'].replace(np.nan, 9).astype(int)

df_dados['ESQ_INI_N_CAT'].value_counts()

### Tratamento variavel CONTREG

In [ ]:
df_dados['CONTREG_CAT'] = df_dados['CONTREG'].replace(np.nan, 0).astype(int)

df_dados['CONTREG_CAT'].value_counts()

### Tratamento variavel NERVOSAFET

In [ ]:
df_dados['NERVOSAFET_CAT'] = df_dados['NERVOSAFET'].replace(np.nan, 0).astype(int)

df_dados['NERVOSAFET_CAT'].value_counts()

### Tratamento variavel AVAL_ATU_N

In [ ]:
df_dados['AVAL_ATU_N_CAT'] = df_dados['AVAL_ATU_N'].replace({np.nan: 9, 'N': 9}).astype(int)

df_dados['AVAL_ATU_N_CAT'].value_counts()

### Tratamento variavel ESQ_ATU_N

In [ ]:
df_dados['ESQ_ATU_N_CAT'] = df_dados['ESQ_ATU_N'].replace({np.nan: 9, 'N': 9}).astype(int)

df_dados['ESQ_ATU_N_CAT'].value_counts()

### Tratamento variavel DOSE_RECEB

In [ ]:
df_dados['DOSE_RECEB_CAT'] = df_dados['DOSE_RECEB'].replace(np.nan, 0).astype(int)

df_dados['DOSE_RECEB_CAT'].value_counts()

### Tratamento variavel EPIS_RACIO

In [ ]:
df_dados['EPIS_RACIO_CAT'] = df_dados['EPIS_RACIO'].replace({np.nan: 9, 'N': 9}).astype(int)

df_dados['EPIS_RACIO_CAT'].value_counts()

### Tratamento variavel CONTEXAM

In [ ]:
df_dados['CONTEXAM_CAT'] = df_dados['CONTEXAM'].replace({np.nan: 0}).astype(int)

df_dados['CONTEXAM_CAT'].value_counts()

### Tratamento variavel TPALTA_N

In [ ]:
df_dados['TPALTA_N_CAT'] = df_dados['TPALTA_N'].replace({np.nan: 9, 'N': 9}).astype(int)

df_dados['TPALTA_N_CAT'].value_counts()

### Drop das variveis sobrando apenas as relevantes

In [15]:
colunas_eliminadas = {"NDUPLIC_N",
                      "ID_UNIDADECS_GESTANT",
                        "SEM_DIAG",
                        "SG_UF",
                        "CS_ESCOL_N",
                        "ID_MN_RESI",
                        "ID_RG_RESI",
                        "ID_PAIS",
                        "NDUPLIC_N",
                        "DT_DIGITA",
                        "DT_TRANSUS",
                        "DT_TRANSDM",
                        "DT_TRANSSM",
                        "DT_TRANSRS",
                        "DT_TRANSSE",
                        "NU_LOTE_V",
                        "NU_LOTE_H",
                        "MIGRADO_W",
                        "ID_OCUPA_N",
                        "IN_VINCULA",
                        "NU_LOTE_IA"
                    }

In [16]:
# Dropar as colunas especificadas
df_dados = df_dados.drop(columns=colunas_eliminadas, errors='ignore')

print(f"As colunas {colunas_eliminadas} foram removidas.")
print(f"O DataFrame agora tem {df_dados.shape[1]} colunas.")

As colunas {'DT_TRANSUS', 'ID_OCUPA_N', 'SEM_DIAG', 'DT_TRANSSM', 'NU_LOTE_V', 'DT_DIGITA', 'NU_LOTE_H', 'ID_MN_RESI', 'ID_RG_RESI', 'DT_TRANSRS', 'NDUPLIC_N', 'DT_TRANSDM', 'MIGRADO_W', 'CS_ESCOL_N', 'ID_PAIS', 'ID_UNIDADECS_GESTANT', 'DT_TRANSSE', 'IN_VINCULA', 'NU_LOTE_IA', 'SG_UF'} foram removidas.
O DataFrame agora tem 45 colunas.


### Colunas numericas

In [17]:
# Selecionar colunas numéricas
numeric_df = df_dados.select_dtypes(include=['number'])
numeric_df.columns

Index(['NU_ANO', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE',
       'NU_IDADE_N', 'CS_GESTANT', 'CS_RACA', 'DT_TRANSRM', 'NU_LESOES',
       'FORMACLINI', 'AVALIA_N', 'CLASSOPERA', 'MODOENTR', 'BACILOSCOP',
       'ESQ_INI_N', 'CONTREG', 'NERVOSAFET', 'ID_MUNI_AT', 'ID_UNID_AT',
       'UFRESAT', 'MUNIRESAT', 'DOSE_RECEB', 'CONTEXAM', 'CS_SEXO_CAT',
       'AVAL_ATU_N_CAT', 'CLASSATUAL_CAT', 'IDADE_ANOS', 'IDADE_MESES'],
      dtype='object')

### Visualização de dados

In [23]:
df_dados[0:-10]

,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_DIAG,ANO_NASC,NU_IDADE_N,CS_SEXO,...,EPIS_RACIO,DTMUDESQ,CONTEXAM,DTALTA_N,TPALTA_N,CS_SEXO_CAT,AVAL_ATU_N_CAT,CLASSATUAL_CAT,IDADE_ANOS,IDADE_MESES
0,2001-01-10,2001,41.0,410304,1359.0,2741474.0,2000-12-10,1942,4058,M,...,NaN,NaN,0.0,2002-10-02,1,1,0,2,58.0,0.0
1,2001-01-17,2001,41.0,410940,1359.0,2741369.0,2000-12-23,1956,4044,M,...,NaN,NaN,2.0,2001-12-19,1,1,0,2,44.0,0.0
2,2001-01-16,2001,41.0,410940,1359.0,2741369.0,2001-01-16,1968,4032,M,...,NaN,NaN,1.0,2001-06-27,1,1,0,1,32.0,0.0
3,2001-01-08,2001,41.0,411780,1359.0,2743116.0,2001-01-08,1960,4040,F,...,NaN,NaN,3.0,2002-01-28,1,2,0,2,40.0,0.0
4,2001-01-02,2001,41.0,411780,1359.0,2743116.0,2000-01-11,1968,4031,M,...,NaN,NaN,0.0,2002-03-14,1,1,2,2,31.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984153,2025-01-21,2025,32.0,320440,32004.0,5569923.0,2024-06-05,1957,4067,M,...,NaN,NaN,NaN,NaN,NaN,1,9,2,67.0,0.0
984154,2025-01-22,2025,32.0,320130,32002.0,2546892.0,2022-11-08,1952,4072,M,...,4,NaN,0.0,NaN,NaN,1,9,2,72.0,0.0
984155,2025-01-10,2025,32.0,320060,32001.0,9784136.0,2025-01-07,1984,4040,M,...,NaN,NaN,NaN,NaN,NaN,1,9,2,40.0,0.0
984156,2025-01-23,2025,32.0,320090,32003.0,2445859.0,2025-01-23,1937,4087,F,...,4,NaN,1.0,NaN,NaN,2,0,1,87.0,0.0
